In [18]:
import pandas as pd
df = pd.read_csv("../data/processed/day3_featured_data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer_flag,Revenue,zero_price_flag,is_return
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30,False,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,False,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00,False,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,False,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,False,False


Create a return risk flag to flag those transactions where orders are returned though they miss or have abnormal entries

In [ ]:
df['return_risk_flag'] = df['is_return'] & (
    df['zero_price_flag'] | df['missing_customer_flag']
)

Creating a risk score based on all the flags that we created to detect frauds or abnormal patterns

In [22]:
df['risk_score'] = (
    df['missing_customer_flag'].astype(int) +   # traceability risk
    df['zero_price_flag'].astype(int) +         # pricing anomaly
    df['return_risk_flag'].astype(int)          # risky returns only
)

In [23]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'missing_customer_flag',
       'Revenue', 'zero_price_flag', 'is_return', 'return_risk_flag',
       'risk_score'],
      dtype='object')

Design an algo which decides whether the transaction is low, medium or high risk!


In [24]:
def risk_category(score):
    if score >= 2:
        return "High Risk"
    elif score == 1:
        return "Medium Risk"
    else:
        return "Low Risk"

df['risk_category'] = df['risk_score'].apply(risk_category)

In [25]:
df['risk_category'].value_counts()

risk_category
Low Risk       401564
Medium Risk    132226
High Risk        2851
Name: count, dtype: int64

In [26]:
df[df['risk_category'] == "High Risk"].head(15)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer_flag,Revenue,zero_price_flag,is_return,return_risk_flag,risk_score,risk_category
605,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1934,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1935,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1936,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1951,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1952,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1984,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1985,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
1986,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom,True,0.0,True,False,False,2,High Risk
2362,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,True,-0.0,True,True,True,3,High Risk


In [27]:
df.to_csv("../data/processed/day4_risk_scored_data.csv", index=False)

## 📌 Day 4: Audit Risk Scoring System

A rule-based audit risk scoring system was developed to identify and prioritize potentially risky transactions.

### 🔍 Key Risk Indicators:
- **Missing Customer Data**: Transactions without CustomerID reduce traceability.
- **Zero or Invalid Pricing**: Transactions with zero or negative prices indicate possible revenue leakage or data issues.
- **Conditional Return Risk**: Return transactions were considered risky only when combined with anomalies such as missing customer data or zero pricing.

### ⚙️ Risk Scoring Logic:
Each transaction was assigned a risk score based on the presence of these indicators:
- 1 point for each risk condition satisfied

### 📊 Risk Categorization:
- **Low Risk (0)**: No significant issues detected  
- **Medium Risk (1)**: One risk indicator present  
- **High Risk (2 or more)**: Multiple risk indicators present  

### 🎯 Outcome:
The system successfully identified a small subset of high-risk transactions characterized by strong anomalies such as missing customer information and zero pricing. This enables auditors to focus on the most critical transactions efficiently.

### 🧠 Key Insight:
Rather than treating all returns as risky, a refined approach was applied to only flag returns when combined with other anomalies, reducing false positives and improving audit accuracy.